<a href="https://colab.research.google.com/github/Ansh1657/MRI-Deepfake-Detection-System/blob/main/notebooks/03_resnet18_orientation_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports & Setup

In [ ]:
"""
ResNet-18 Orientation Classifier Training
Trains a model to classify MRI scans into Axial, Coronal, or Sagittal planes.
Includes data reorganization, training loops, and an automated sorting script.
"""

import os
import shutil
import random
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from PIL import Image

Dataset Reorganization & Splitting

In [ ]:
# 1. Define paths
dark_dir = '../data/raw_mri/T1_Dark'
bright_dir = '../data/raw_mri/FLAIR_Bright'
output_dir = '../data/processed_256/orientation_dataset'

classes = ['axial', 'coronal', 'sagitall']

print("Creating PyTorch folder structure...")

# 2. Create the empty train/val structure
for split in ['train', 'val']:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)

# 3. Function to pool, shuffle, and split files
def process_and_split(class_name):
    all_files = []

    dark_path = os.path.join(dark_dir, class_name)
    if os.path.exists(dark_path):
        all_files.extend([os.path.join(dark_path, f) for f in os.listdir(dark_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    bright_path = os.path.join(bright_dir, class_name)
    if os.path.exists(bright_path):
        all_files.extend([os.path.join(bright_path, f) for f in os.listdir(bright_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

    random.shuffle(all_files)

    split_idx = int(len(all_files) * 0.8)
    train_files = all_files[:split_idx]
    val_files = all_files[split_idx:]

    for f in train_files:
        folder_origin = "dark_" if dark_dir in f else "bright_"
        new_name = folder_origin + os.path.basename(f)
        shutil.copy(f, os.path.join(output_dir, 'train', class_name, new_name))

    for f in val_files:
        folder_origin = "dark_" if dark_dir in f else "bright_"
        new_name = folder_origin + os.path.basename(f)
        shutil.copy(f, os.path.join(output_dir, 'val', class_name, new_name))

    print(f"{class_name.capitalize()}: {len(train_files)} Train images, {len(val_files)} Validation images.")

# 4. Execute Split (Commented out to protect IP on GitHub)
# for cls in classes:
#     process_and_split(cls)
# print(f"\nDone! Your dataset is ready for training at: {output_dir}")

ResNet-18 Training Loop

In [ ]:
# 1. Setup Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Define Data Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# 3. Load the Dataset
data_dir = '../data/processed_256/orientation_dataset'

try:
    image_datasets = {x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x]) for x in ['train', 'val']}
    dataloaders = {x: DataLoader(image_datasets[x], batch_size=16, shuffle=True, num_workers=2) for x in ['train', 'val']}
    dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
    class_names = image_datasets['train'].classes
    print(f"Classes found: {class_names}")

    # 4. Initialize Pre-trained ResNet18
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, len(class_names))
    model = model.to(device)

    # 5. Define Loss and Optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

    # 6. Training Loop
    num_epochs = 10
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    print("\nStarting Training...")
    since = time.time()

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best Validation Accuracy: {best_acc:4f}')

    # Save best model weights
    model.load_state_dict(best_model_wts)
    save_path = '../saved_models/resnet18_router.pth'
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    torch.save(model.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

except FileNotFoundError:
    print("Warning: Dataset directory not found. Execution bypassed to protect IP.")

Automated Sorting (Inference)

In [ ]:
# 1. Setup Device and Classes
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
class_names = ['axial', 'coronal', 'sagitall']

# 2. Rebuild the Model Architecture and Load Weights
print("Loading model for automated sorting...")
model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))

model_path = '../saved_models/resnet18_router.pth'

if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    model = model.to(device)
    model.eval()
    print("Model loaded successfully!")

    # 3. Vision transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # 4. Define Folders
    input_folder = '../data/raw_mri/unsorted_FLAIR'
    output_base_folder = '../data/processed_256/sorted_FLAIR'

    if os.path.exists(input_folder):
        for cls in class_names:
            os.makedirs(os.path.join(output_base_folder, cls), exist_ok=True)

        # 5. Sorting Loop
        print(f"Starting automatic sorting from: {input_folder}")
        processed_count = 0
        image_files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        with torch.no_grad():
            for filename in image_files:
                img_path = os.path.join(input_folder, filename)
                try:
                    img = Image.open(img_path).convert('RGB')
                    img_tensor = transform(img).unsqueeze(0).to(device)

                    outputs = model(img_tensor)
                    _, predicted_idx = torch.max(outputs, 1)
                    predicted_class = class_names[predicted_idx.item()]

                    dest_path = os.path.join(output_base_folder, predicted_class, filename)
                    shutil.copy(img_path, dest_path)

                    processed_count += 1
                    if processed_count % 100 == 0:
                        print(f"Sorted {processed_count} images...")

                except Exception as e:
                    print(f"Skipping {filename} due to error: {e}")

        print(f"\nSuccess! Automatically sorted {processed_count} images into {output_base_folder}")
    else:
        print("Warning: Unsorted input folder not found. Execution bypassed.")
else:
    print("Warning: Trained weights not found. Execute the training cell first.")